# 📊 The Data Lineage Inspector & Feedback Loop Simulator
**MSU AI Club — Fall Series Kickoff (9/14/2026)**  
*Today's Track: Billy Joel — Vienna | Event Link: [MSU AI Club Event Page](https://www.msuaiclub.com/events)*

---

### Part 1: Loading the Pipeline & Inspecting Raw Data [2]
In this section, we load the raw credit and scholarship applicant dataset (`applicant_data_raw.csv`).
Your job is to inspect the raw feature matrix and identify what critical human dimensions are missing before any algorithm ever touches the data [2].

In [ ]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
import plotly.express as px
import plotly.graph_objects as go
import os

# Load dataset locally or from GitHub raw backup if running in Colab
DATA_URL = "applicant_data_raw.csv"
if not os.path.exists(DATA_URL):
    DATA_URL = "https://raw.githubusercontent.com/lowellmonis/msu-ai-workshops/main/workshops/workshop1/applicant_data_raw.csv"

df = pd.read_csv(DATA_URL)

print("=" * 60)
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print("=" * 60)
print("\nFirst 5 rows of applicant dataset:")
display(df.head())

print("\nMissing values breakdown by column:")
print(df.isnull().sum())

# STUDENT TASK 1:
# Look at the raw columns above. Write a 1-sentence comment answering:
# What variables are completely missing from this dataset that might matter for an applicant? [2]

student_reflection_q1 = """
# TODO: Replace this string with your reflection
# E.g., 'The dataset lacks non-traditional financial metrics like rental payment history, caregiving duties, and gig-economy stability [2].'
"""
print("\nYour Reflection Q1:")
print(student_reflection_q1)

### Part 2: The Editing Room (Data Cleaning Audit) [2]
Data cleaning is not a neutral technical step — it is an editorial choice [2].
Watch what happens to demographic representation when we apply standard `df.dropna()` [2].

In [ ]:
# --- Pre-written Naïve Cleaning Script ---
df_naive = df.dropna()

dropped_count = len(df) - len(df_naive)
print(f"Naïve dropna() removed {dropped_count} rows ({dropped_count/len(df):.1%} of the dataset).\n")

audit_comparison = pd.DataFrame({
    "Original Count": df["group"].value_counts(),
    "After dropna()": df_naive["group"].value_counts(),
    "Percent Lost": ((df["group"].value_counts() - df_naive["group"].value_counts()) / df["group"].value_counts() * 100).round(1)
})
print("--- Demographics of Dropped Applicants ---")
display(audit_comparison)

# --- STUDENT CODE CHALLENGE ---
# Complete the missing line below to test an alternative imputation method (median fill for numeric columns)
# instead of outright deletion!

# TODO: Fill missing numeric columns with the median instead of dropping them
df_cleaned = df.fillna(df.median(numeric_only=True))

# Verify imputation
print("\nMissing values after median imputation:")
print(df_cleaned[["credit_score", "years_employed"]].isnull().sum())
print(f"Total rows preserved with median imputation: {len(df_cleaned)}")

### Part 3: The Black Box & Disparate Impact (O'Neil's WMD Framework) [1]
Now we train a `DecisionTreeClassifier` on the cleaned dataset to predict applicant approvals.
We will measure **Disparate Impact Ratio** across demographic groups [1]:
$$\text{Disparate Impact Ratio} = \frac{\text{Approval Rate of Group B}}{\text{Approval Rate of Group A}}\\
Under the legal 4/5ths (80%) rule, a ratio below 0.80 indicates adverse impact [1].

In [ ]:
# Train baseline model
features = ["income", "credit_score", "years_employed", "debt_to_income"]
X = df_cleaned[features]
y = df_cleaned["historical_approval"]

model = DecisionTreeClassifier(max_depth=4, random_state=42)
model.fit(X, y)

df_cleaned["predicted_approval"] = model.predict(X)

# Calculate acceptance rates across demographic segments
group_stats = df_cleaned.groupby("group")["predicted_approval"].agg(["count", "mean"]).reset_index()
group_stats.columns = ["group", "total_applicants", "approval_rate"]

rate_A = group_stats.loc[group_stats["group"] == "Group A", "approval_rate"].values[0]
rate_B = group_stats.loc[group_stats["group"] == "Group B", "approval_rate"].values[0]
disparate_impact_ratio = rate_B / rate_A

print(f"Group A Approval Rate: {rate_A:.1%}")
print(f"Group B Approval Rate: {rate_B:.1%}")
print(f"Disparate Impact Ratio (Group B / Group A): {disparate_impact_ratio:.3f}")

if disparate_impact_ratio < 0.80:
    print("⚠️ ADVERSE IMPACT DETECTED: Ratio is below the 80% legal threshold (O'Neil WMD Warning!)")

# Plotly visual
fig = px.bar(
    group_stats,
    x="group",
    y="approval_rate",
    color="group",
    color_discrete_map={"Group A": "#08ffff", "Group B": "#ff0055"},
    title=f"Disparate Impact Ratio: {disparate_impact_ratio:.2f} (Threshold = 0.80)",
    labels={"approval_rate": "Predicted Approval Rate", "group": "Applicant Cohort"},
    text_auto=".1%"
)
fig.add_hline(y=rate_A * 0.80, line_dash="dash", line_color="yellow", annotation_text="80% Disparate Impact Threshold")
fig.update_layout(template="plotly_dark", font_family="Rubik")
fig.show()

# STUDENT TASK 2:
# Identify which group experiences the highest negative classification error and write a short note
# explaining 'opacity' and 'damage' based on Cathy O'Neil's WMD framework [1].
student_reflection_q2 = """
# TODO: Write your note on Opacity and Damage below
# E.g., 'Group B suffers from lower approval due to unmeasured non-traditional signals (Damage). The black box classifier hides feature weighting (Opacity) [1].'
"""
print("\nYour Reflection Q2:")
print(student_reflection_q2)

### Part 4: Closing the Loop (Recursive Bias Simulation) [1]
Models deployed in the real world generate outcomes that become next year's training data [1].
In this simulation, high-confidence model approvals from Generation 1 are fed back as ground-truth labels for Generation 2 over 3 iterations [1].

In [ ]:
def simulate_feedback_loop(df_base, generations=3, audit_rate=0.0):
    history = []
    current_df = df_base.copy()
    
    for gen in range(1, generations + 1):
        X_gen = current_df[features]
        y_gen = current_df["historical_approval"]
        
        clf = DecisionTreeClassifier(max_depth=4, random_state=gen)
        clf.fit(X_gen, y_gen)
        
        preds = clf.predict(X_gen)
        
        # Apply Human Review Audit Sampling Override if configured
        if audit_rate > 0.0:
            rejection_indices = np.where(preds == 0)[0]
            num_audited = int(len(rejection_indices) * audit_rate)
            audited_samples = np.random.choice(rejection_indices, size=num_audited, replace=False)
            # Human oversight rescues qualified non-traditional applicants
            preds[audited_samples] = (current_df.iloc[audited_samples]["income"] > 45000).astype(int)
            
        current_df["predicted_approval"] = preds
        
        rate_a = current_df[current_df["group"] == "Group A"]["predicted_approval"].mean()
        rate_b = current_df[current_df["group"] == "Group B"]["predicted_approval"].mean()
        di_ratio = rate_b / rate_a if rate_a > 0 else 0
        
        history.append({
            "Generation": f"Gen {gen}",
            "Group A Approval": rate_a,
            "Group B Approval": rate_b,
            "Disparate Impact Ratio": di_ratio,
            "Audit Rate": f"{int(audit_rate*100)}%"
        })
        
        # Feedback loop: model predictions become ground truth for next generation
        current_df["historical_approval"] = current_df["predicted_approval"]
        
    return pd.DataFrame(history)

# Baseline feedback loop without human review
history_unmitigated = simulate_feedback_loop(df_cleaned, generations=4, audit_rate=0.0)

# STUDENT CODE CHALLENGE:
# Write a 2-line override filter to inject human review audit sampling (forcing 10% of automated rejections to be manually reviewed)
# and observe how it dampens the feedback loop curve!

# TODO: Implement a manual review override for bottom-tier risk scores
history_mitigated = simulate_feedback_loop(df_cleaned, generations=4, audit_rate=0.10)

combined_history = pd.concat([
    history_unmitigated.assign(Mode="Unmitigated (Automated Loop)"),
    history_mitigated.assign(Mode="Mitigated (10% Human Audit)")
])

fig_loop = px.line(
    combined_history,
    x="Generation",
    y="Disparate Impact Ratio",
    color="Mode",
    markers=True,
    title="Feedback Loop Dynamics: Unmitigated Cascade vs. 10% Human Review Audit",
    labels={"Disparate Impact Ratio": "Disparate Impact Ratio (Group B / Group A)"},
    color_discrete_map={"Unmitigated (Automated Loop)": "#ff0055", "Mitigated (10% Human Audit)": "#08ffff"}
)
fig_loop.add_hline(y=0.80, line_dash="dash", line_color="yellow", annotation_text="80% Parity Threshold")
fig_loop.update_layout(template="plotly_dark", font_family="Rubik")
fig_loop.show()

### Part 5: Saving Your Data Lifecycle Map & References
Congratulations! You have completed the data lineage audit and feedback loop simulation.
Run the cell below to export your code reflections and simulation outputs into a clean Markdown summary (`audit_summary.md`).
This document forms **Page 1 of your MSU AI Club Portfolio**!

In [ ]:
def export_audit_summary(file_name="audit_summary.md"):
    content = f"""# 📜 Portfolio Artifact: Data Lineage & Feedback Loop Audit
**Author / Student**: MSU AI Club Member  
**Workshop Date**: September 14, 2026  
**Event**: Kickoff Workshop — *The Art of the Data Lifecycle: Storytelling in AI*  
**Song of the Day**: Billy Joel — *Vienna*  

---

## 1. Raw Data Inspection & Unmeasured Variables [2]
> **Reflection Question**: What variables are completely missing from this dataset that might matter for an applicant?  
> **Student Answer**:  
> {student_reflection_q1.strip()}

---

## 2. Data Cleaning Audit (Naïve dropna vs Imputation) [2]
- **Total Applicants**: {len(df)}
- **Naïve Deletion Loss**: {dropped_count} rows dropped ({dropped_count/len(df):.1%})
- **Demographic Impact**: Non-traditional Group B lost disproportionately more rows due to missing credit reporting [2].
- **Imputation Remedy**: Applied median fill (`df.fillna(df.median(numeric_only=True))`), preserving 100% of applicants.

---

## 3. Disparate Impact & WMD Analysis [1]
- **Group A Approval Rate**: {rate_A:.1%}
- **Group B Approval Rate**: {rate_B:.1%}
- **Disparate Impact Ratio**: {disparate_impact_ratio:.3f} (Legal Threshold: 0.80) [1]
> **WMD Framework Reflection (Opacity & Damage)**:  
> {student_reflection_q2.strip()}

---

## 4. Feedback Loop Mitigation [1]
- **Unmitigated Feedback Loop**: Disparate impact degrades over 4 generations as automated predictions reinforce historical exclusion [1].
- **Human Oversight Intervention**: Adding a 10% manual audit sample on rejections prevents catastrophic feedback cascades and stabilizes parity above the 0.80 threshold.

---

## 5. Personal Ethical Commitment [1]–[4]
*I commit to inspecting upstream data collection choices, auditing cleaning filters for demographic loss, and advocating for human oversight loops in automated decision systems.*

---

## 6. References
[1] C. O'Neil, Weapons of Math Destruction: How Big Data Increases Inequality and Threatens Democracy. Crown, 2016.
[2] C. D'Ignazio and L. F. Klein, Data Feminism. MIT Press, 2020.
[3] Pope Leo XIV, Magnifica Humanitas: On Safeguarding the Human Person in the Time of AI, 2025.
[4] E. Yudkowsky and N. Soares, If Anyone Builds It, Everyone Dies. Little, Brown and Company, 2025.
"""
    with open(file_name, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"✅ Success! Your audit summary has been saved to: {file_name}")
    print("Download this file or push it to your portfolio GitHub repository!")

export_audit_summary()

## 📚 References
[1] C. O'Neil, *Weapons of Math Destruction: How Big Data Increases Inequality and Threatens Democracy*. New York, NY, USA: Crown Publishing Group, 2016.

[2] C. D'Ignazio and L. F. Klein, *Data Feminism*. Cambridge, MA, USA: MIT Press, 2020.

[3] Pope Leo XIV, *Magnifica Humanitas: On Safeguarding the Human Person in the Time of Artificial Intelligence*, Encyclical Letter, Vatican City: Libreria Editrice Vaticana, 2025.

[4] E. Yudkowsky and N. Soares, *If Anyone Builds It, Everyone Dies: Why Superhuman AI Would Kill Us All*. New York, NY, USA: Little, Brown and Company, 2025.

